# VECTOR TILING FOR MAP GENERALIZATION ETC

Generalization means creating many copies of a single ‘source of truth’ and tailoring them to the view context

Doing this on the fly is labor intensive in a static paper mapping context, especially en masse with variable map scales and contexts (urban, semi-urban, rural) 

Value in adopting dynamic mapping techniques into the existing processing workflow:

- Vector tiling: highly optimized, customizable, retains records’ attributes (versus image tiles)
- Customization, filtering, alignment between 'in-house' GRID3 data and third-party sources like OpenStreetMap
- API endpoint targets: maps can diff and grow as the sources of truth do
- TIPPECANOE: Incredible and fast processing library, somewhat unpredictable w/ emergent effects when processing geometries in a ‘recursive tiled’ manner, but benefits outweigh headaches
- Plus: Self-hosting interest (docker/containerization): anxiety after ‘open’ data portals going offline nationwide



<!-- - What works?

- What doesn't? -->
<!-- 
Note: “Emergent properties” -> usually helpful automation, as long as there are constraints, but need to watch for weirdness and misalignment with what a legend would otherwise say a layer/category should look like

(For instance, interpolation between color steps, etc) -->

===

# Processing pipeline

1. **Download** - Fetch Overture Maps and GRID3 (AGOL feature services) data for specified extent (as GeoParquet file)
2. **Convert to FlatGeobuf** - Transform GeoParquet to FlatGeobuf for compatibility + efficiency
3. **Tile** - Generate PMTiles using tippecanoe with bespoke settings per-layer
4. **View** - using maplibre open spec

## File formats
- **GeoParquet (.parquet)** - Download format (compact, fast queries via "duckquery")
- **FlatGeobuf (.fgb)** - Convert for optimal tippecanoe library support
- **GeoJSON (.geojson)** - Fallback support for small datasets
- **pmTiles** - Dynamic vector tiles, served from single static file

## CONFIG

.env -> config.py imports environment vars

In [1]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Setup paths
notebook_dir = Path.cwd()
print(f"Current notebook directory: {notebook_dir}")
processing_dir = notebook_dir  # preprocessing
repo_root = notebook_dir       # repository root

if str(processing_dir) not in sys.path:
    sys.path.insert(0, str(processing_dir))

# Load environment variables from repository root .env
env_path = repo_root / '.env'
load_dotenv(env_path)
print(f"✓ Loaded environment from: {env_path}")
print(f"  DATA_DISK = {os.environ.get('DATA_DISK', 'not set')}")

# ── PROJ library fix for GDAL/OGR on WSL2 ──────────────────────────────────
proj_lib_path = Path.home() / "micromamba/envs/geoprocessing/share/proj"
if proj_lib_path.exists():
    os.environ['PROJ_LIB'] = str(proj_lib_path)
    print(f"✓ Set PROJ_LIB: {proj_lib_path}")
else:
    conda_prefix = os.environ.get('CONDA_PREFIX', '')
    if conda_prefix:
        alt_proj = Path(conda_prefix) / "share/proj"
        if alt_proj.exists():
            os.environ['PROJ_LIB'] = str(alt_proj)
            print(f"✓ Set PROJ_LIB: {alt_proj}")
        else:
            print(f"!!!  Warning: PROJ data not found at {proj_lib_path}")
    else:
        print(f"!!!  Warning: PROJ data not found — GDAL may have issues")

# ── Import configuration ────────────────────────────────────────────────────
from config import (
    get_config,
    ensure_directories,
    print_config_summary,
    SCRIPTS_DIR,
    SCRATCH_DIR,
    SCRATCH_GRID3_DIR,
    OUTPUT_DIR,
    OUTPUT_GRID3_DIR,
    GRID3_INPUT_DIR,
    OVERTURE_INPUT_DIR,
    SCRATCH_OVERTURE_DIR,
    MIGRATION_TARGET_DIR,
    GRID3_ISO3_CODES,
    # Per-country path helpers
    grid3_input,
    grid3_scratch,
    grid3_scratch_filtered,
    grid3_scratch_temp,
    grid3_output,
)

# Import processing functions
from scripts import (
    download_overture_data,
    download_overture_buildings_cli,
    test_service_connection,
    convert_file,
    convert_parquet_to_fgb,
    batch_generate_centroids,
    batch_convert_directory,
    convert_geodata_to_fgb,
    batch_convert_geodata,
    convert_gpkg_to_fgb_layers,
    # Tiling — four targeting modes:
    process_to_tiles,    # (d) all data (full run)
    process_group,       # (a) single LAYER_GROUP for one ISO3
    process_iso3,        # (c) all groups for one country
    process_theme,       # (b) one theme across all countries
    create_tilejson,
    download_arcgis_data,
    batch_download_arcgis_layers,
    filter_fgb as _filter_fgb,
    _resolve_filter,
    run_merge_config,
)

import pandas as pd
import polars as pl
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Global config ────────────────────────────────────────────────────────────
CONFIG = get_config()

# ── Extent from .env ─────────────────────────────────────────────────────────
print(f"\n=== EXTENT FROM ENVIRONMENT (.env) ===")
print(f"  West:   {os.environ.get('EXTENT_WEST', 'not set')}")
print(f"  South:  {os.environ.get('EXTENT_SOUTH', 'not set')}")
print(f"  East:   {os.environ.get('EXTENT_EAST', 'not set')}")
print(f"  North:  {os.environ.get('EXTENT_NORTH', 'not set')}")
print(f"  Buffer: {os.environ.get('EXTENT_BUFFER', '0.0')} degrees")
print(f"  Tuple:  {CONFIG['extent']['coordinates']}")

# ── Processing options ────────────────────────────────────────────────────────
CONFIG["fgb_conversion"]["cleanup_source"] = False  # keep source .parquet files
CONFIG["download"]["verbose"] = True
CONFIG["conversion"]["verbose"] = True
CONFIG["tiling"]["verbose"] = True
CONFIG["tiling"]["parallel"] = False

# ── Tiling profile ────────────────────────────────────────────────────────────
# "iso3_theme" — one PMTiles per country+theme → 3-pmtiles/grid3/{iso3}/
# "iso3"       — one PMTiles per country (tile-join merge) → 3-pmtiles/grid3/GRID3_{ISO3}.pmtiles
# "all"        — single global archive → 3-pmtiles/grid3/GRID3.pmtiles
CONFIG["tiling"]["profile"] = "iso3"
CONFIG["tiling"]["keep_theme_files"] = True

# Create directories and verify
ensure_directories()

print("\n=== CONFIGURATION VERIFICATION ===")
print(f"DATA_DISK:            {os.environ.get('DATA_DISK', 'NOT SET')}")
print(f"Scratch (grid3):      {SCRATCH_GRID3_DIR}")
print(f"Output (grid3):       {OUTPUT_GRID3_DIR}")
print(f"Migration target:     {MIGRATION_TARGET_DIR}")
print(f"ISO3 codes:           {GRID3_ISO3_CODES}")
print_config_summary(CONFIG)

print(f"\n=== TILING PROFILE ===")
print(f"  Profile:          {CONFIG['tiling']['profile']}")
print(f"  Keep theme files: {CONFIG['tiling']['keep_theme_files']}")
print(f"  Output root:      {CONFIG['tiling']['output_dir']}")

print("\n✓ Configuration loaded — tiling shortcuts available:")
print("  process_group('GRID3_COD_boundaries')   < - (a) single group / one ISO3")
print("  process_theme('boundaries')              < - (b) one theme / all countries")
print("  process_iso3('cod')                      < - (c) all groups / one country")
print("  process_to_tiles(...)                    < - (d) full run — see Step 4")

Current notebook directory: /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing
✓ Loaded environment from: /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing/.env
  DATA_DISK = /tmp/grid3_tiles
✓ Set PROJ_LIB: /home/mjh2241/micromamba/envs/geoprocessing/share/proj

=== EXTENT FROM ENVIRONMENT (.env) ===
  West:   -18.1
  South:  -35.1
  East:   51.6
  North:  37.4
  Buffer: 1 degrees
  Tuple:  (-18.1, -35.1, 51.6, 37.4)

=== CONFIGURATION VERIFICATION ===
DATA_DISK:            /tmp/grid3_tiles
Scratch (grid3):      /tmp/grid3_tiles/data/2-scratch/grid3
Output (grid3):       /tmp/grid3_tiles/data/3-pmtiles/grid3
Migration target:     /mnt/d/mheaton/grid3_tiles/data/3-pmtiles
ISO3 codes:           ['cod', 'nga', 'africa']
PROJECT CONFIGURATION
Project root:         /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing
Data disk:            /tmp/grid3_tiles
Input (grid3):        /tmp/grid3_tiles/data/1-input/grid3
Input (overture):     /tmp/grid3_tiles/data/1-input/overture
Scratch (grid3):     

## Optional: Download Overture Buildings

Use the `overturemaps` Python package (primary) or the legacy DuckDB path to fetch Overture buildings for the configured extent.

e.g., replaces periodic geofabrik OSM shapefile fetches

In [ ]:
# Download Overture buildings via overturemaps Python package (primary / recommended)
print("=== STEP 1a: DOWNLOADING OVERTURE BUILDINGS (overturemaps) ===")
overture_results = download_overture_buildings_cli(
    extent=CONFIG["extent"]["coordinates"],
    output_dir=str(OVERTURE_INPUT_DIR),
    verbose=CONFIG["download"]["verbose"],
)
print(f"Download completed: {overture_results['success']}")
if overture_results["output_file"]:
    print(f"Output: {overture_results['output_file']}")
if overture_results["error"]:
    print(f"Error: {overture_results['error']}")

In [ ]:
# Alternative: Download Overture Maps data via DuckDB (legacy)
print("=== STEP 1b: DOWNLOADING OVERTURE DATA (DuckDB legacy) ===")
download_results = download_overture_data(
    extent=CONFIG["extent"]["coordinates"],
    buffer_degrees=CONFIG["extent"]["buffer_degrees"],
    template_path=str(CONFIG["paths"]["template_path"]),
    verbose=CONFIG["download"]["verbose"],
    project_root=str(CONFIG["paths"]["project_root"]),
    overture_data_dir=str(OVERTURE_INPUT_DIR),
    duckdb_temp_dir=str(CONFIG["paths"]["duckdb_temp_dir"]),
)
print(f"Download completed: {download_results['success']}")
print(f"Sections processed: {download_results['processed_sections']}")
if download_results["errors"]:
    print(f"Errors encountered: {len(download_results['errors'])}")
    for error in download_results["errors"]:
        print(f"  - {error}")
print()

## Optional: Download ArcGIS Feature Server Data

Download geospatial data from hosted ArcGIS Feature Server REST API endpoints - can include any esri-hosted data as endpoint

- **Automatic pagination** - Handles ArcGIS's 1000-2000 feature limit per request
- **Spatial filtering** - Apply bounding box filter to download only features in aoi
- **formats** - Download as GeoJSON or directly convert to FlatGeobuf
- **Batch processing** - Download multiple layers with one function call

### GRID3 DRC Layers
- https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services

In [ ]:
# DIAGNOSTIC: Test ArcGIS Feature Server connections before downloading
print("=== DIAGNOSTIC: TESTING ARCGIS SERVICE CONNECTIONS ===\n")


# Test each service URL for accessibility and metadata
# NOTE: Services updated from v7_0 to v8_0 (January 2025)
test_urls = [
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_NGA_settlement_extents_v4_0/FeatureServer/0'
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_health_areas_v8_0/FeatureServer/0',
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_settlement_names_v8_0/FeatureServer/0',
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/COD_GRID3_health_facilities_v8_0/FeatureServer/0'
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_COD_Settlement_Extents_v3_1/FeatureServer/0',
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_NGA_settlement_extents_v4_0/FeatureServer/0'
]

test_results = []
for url in test_urls:
    layer_name = url.split('/services/')[-1].split('/')[0]
    print(f"\nTesting: {layer_name}")
    print("-" * 70)
    
    result = test_service_connection(url, verbose=True)
    test_results.append({
        'layer': layer_name,
        'url': url,
        **result
    })

# Summary
print(f"\n{'='*70}")
print("CONNECTION TEST SUMMARY")
print(f"{'='*70}")
accessible_count = sum(1 for r in test_results if r['accessible'])
print(f"Total services tested: {len(test_results)}")
print(f"Accessible: {accessible_count}")
print(f"Failed: {len(test_results) - accessible_count}")

if accessible_count < len(test_results):
    print(f"\nFailed services:")
    for r in test_results:
        if not r['accessible']:
            print(f"  ✗ {r['layer']}")
            print(f"    Error: {r['error']}")
    
    print(f"\n!!!  TROUBLESHOOTING TIPS:")
    print(f"  1. Check if you need to authenticate (VPN, organizational login)")
    print(f"  2. Verify the service URLs are still active")
    print(f"  3. Check if the organization changed permissions")
    print(f"  4. Try accessing the URL in a browser to see the error message")
else:
    print(f"\n✓ All services accessible - ready to download!")
    print(f"\n📝 SERVICE METADATA:")
    for r in test_results:
        if r['accessible']:
            print(f"\n  {r['layer']}:")
            if 'feature_count' in r:
                print(f"    Features: {r['feature_count']:,}")
            if 'geometry_type' in r:
                print(f"    Geometry: {r['geometry_type']}")
            if 'max_record_count' in r:
                print(f"    Max records/request: {r['max_record_count']:,}")

print(f"{'='*70}\n")


### ArcGIS Feature Server Downloads

**Finding Feature Server URLs:**
1. Browse your organization's ArcGIS REST Services Directory
2. Navigate to a specific layer (e.g., FeatureServer/0, FeatureServer/1)
3. Copy the full URL up to and including the layer number
4. The script will automatically append `/query` and handle parameters

**Spatial Filtering:**
- The `extent` parameter filters features to your bounding box (saves bandwidth & time)
- For global layers, omit `extent=None` to download all features
- Extent uses WGS84 coordinates: `(lon_min, lat_min, lon_max, lat_max)`

**Attribute Filtering:**
- Use `where` clause for SQL-based filtering: `'population > 10000'`
- Default `'1=1'` downloads all features

**Output Formats:**
- `"fgb"` (FlatGeobuf) - Recommended for direct tiling (streaming, indexed)
- `"geojson"` - more flexible, less optimal


- Large datasets (>100k features) automatically use pagination
- Downloads directly to scratch directory

In [ ]:
# Download ArcGIS Feature Server data (optional - skip if not needed)
print("=== STEP 2a: DOWNLOADING ARCGIS DATA  ===")
print(f"Current extent from CONFIG: {CONFIG['extent']['coordinates']}")
print(f"  Longitude: {CONFIG['extent']['coordinates'][0]} to {CONFIG['extent']['coordinates'][2]}")
print(f"  Latitude: {CONFIG['extent']['coordinates'][1]} to {CONFIG['extent']['coordinates'][3]}")

# Option 1: NO SPATIAL FILTER (recommended for large extents — avoids 400 errors)
download_extent = None
# Option 2: USE SPATIAL FILTER
# download_extent = CONFIG["extent"]["coordinates"]

print(f"\nSpatial filtering: {'DISABLED (downloading all features)' if download_extent is None else f'ENABLED {download_extent}'}")

# ── COD BOUNDARIES ────────────────────────────────────────────────────────────
# Admin/operational boundary polygons → 2-scratch/grid3/cod/
cod_boundaries = [
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_COD_health_zones_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_health_zones_v8_0',
    #     'where': '1=1'
    # },
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_health_areas_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_health_areas_v8_0',
    #     'where': '1=1'
    # },
]

if cod_boundaries:
    print(f"\n--- Downloading {len(cod_boundaries)} COD boundary layer(s) → {grid3_scratch('cod')}/ ---")
    b_results = batch_download_arcgis_layers(
        layer_configs=cod_boundaries,
        output_dir=str(grid3_scratch("cod")),
        extent=download_extent,
        output_format="fgb",
        verbose=CONFIG["download"]["verbose"],
        timeout=180
    )
    for layer in b_results['layers']:
        status = f"✓ {layer['name']}: {layer['feature_count']:,} features" if layer['success'] else f"✗ {layer['name']}: {layer.get('error', 'unknown error')}"
        print(f"  {status}")

# ── NGA SETTLEMENT EXTENTS ────────────────────────────────────────────────────
# Settlement extent polygons → 2-scratch/grid3/nga/
nga_extents = [
    {
        'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_NGA_settlement_extents_v4_0/FeatureServer/0',
        'name': 'GRID3_NGA_settlement_extents_v4_0',
        'where': '1=1'
    },
]

if nga_extents:
    print(f"\n--- Downloading {len(nga_extents)} NGA settlement extent layer(s) → {grid3_scratch('nga')}/ ---")
    e_results = batch_download_arcgis_layers(
        layer_configs=nga_extents,
        output_dir=str(grid3_scratch("nga")),
        extent=download_extent,
        output_format="fgb",
        verbose=CONFIG["download"]["verbose"],
        timeout=180
    )
    for layer in e_results['layers']:
        status = f"✓ {layer['name']}: {layer['feature_count']:,} features" if layer['success'] else f"✗ {layer['name']}: {layer.get('error', 'unknown error')}"
        print(f"  {status}")

# ── COD POIs ──────────────────────────────────────────────────────────────────
# Health facilities, settlement names → 2-scratch/grid3/cod/
cod_pois = [
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/COD_GRID3_health_facilities_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_health_facilities_v8_0',
    #     'where': '1=1'
    # },
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_settlement_names_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_settlement_names_v8_0',
    #     'where': '1=1'
    # },
]

if cod_pois:
    print(f"\n--- Downloading {len(cod_pois)} COD POI layer(s) → {grid3_scratch('cod')}/ ---")
    p_results = batch_download_arcgis_layers(
        layer_configs=cod_pois,
        output_dir=str(grid3_scratch("cod")),
        extent=download_extent,
        output_format="fgb",
        verbose=CONFIG["download"]["verbose"],
        timeout=180
    )
    for layer in p_results['layers']:
        status = f"✓ {layer['name']}: {layer['feature_count']:,} features" if layer['success'] else f"✗ {layer['name']}: {layer.get('error', 'unknown error')}"
        print(f"  {status}")

if not any([cod_boundaries, nga_extents, cod_pois]):
    print("\nNo ArcGIS layers configured. Uncomment entries above to download data.")

#### note - spatial filtering is disabled by default when downloading from arcgis feature services. very large extents may trigger a 400 error. when in doubt, download the full extent and post-process (below) to an area of interest

In [ ]:
# OPTIONAL: Clip downloaded features to extent if needed

# import geopandas as gpd
# from pathlib import Path

# print("=== CLIPPING ARCGIS DATA TO EXTENT ===\n")

# # Get extent from CONFIG
# lon_min, lat_min, lon_max, lat_max = CONFIG["extent"]["coordinates"]
# print(f"Clipping extent: ({lon_min}, {lat_min}, {lon_max}, {lat_max})")

# # List of layers to clip
# layers_to_clip = ['settlement_extents']  # Add more as needed

# for layer_name in layers_to_clip:
#     input_file = CONFIG["paths"]["scratch_dir"] / f"{layer_name}.fgb"
    
#     if input_file.exists():
#         print(f"\nProcessing: {layer_name}")
        
#         # Read the file
#         gdf = gpd.read_file(input_file)
#         original_count = len(gdf)
#         print(f"  Original features: {original_count:,}")
        
#         # Clip to extent using GeoPandas spatial indexing
#         gdf_clipped = gdf.cx[lon_min:lon_max, lat_min:lat_max]
#         clipped_count = len(gdf_clipped)
#         print(f"  After clipping: {clipped_count:,}")
#         print(f"  Removed: {original_count - clipped_count:,} features outside extent")
        
#         # Overwrite the original file (or save with different name)
#         gdf_clipped.to_file(input_file, driver="FlatGeobuf")
#         print(f"  ✓ Updated: {input_file.name}")
#     else:
#         print(f"\n✗ {layer_name}.fgb not found")

# print("\n✓ Clipping complete!")


### progress/sanity check

In [ ]:
# Check what files were created during download / conversion
print("=== CHECKING FILES ===")

# Overture inputs (GeoParquet / GeoJSON)
overture_files = []
if OVERTURE_INPUT_DIR.exists():
    for pattern in CONFIG["download"]["output_formats"]:
        overture_files.extend(OVERTURE_INPUT_DIR.glob(pattern))

print(f"\nOverture inputs ({OVERTURE_INPUT_DIR.name}/): {len(overture_files)} files")
overture_size_mb = sum(f.stat().st_size / 1024 / 1024 for f in overture_files)
for f in sorted(overture_files):
    print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

# FlatGeobuf files per ISO3 scratch dir
print(f"\nFlatGeobuf (2-scratch/grid3/):")
fgb_total, fgb_size_mb = 0, 0.0
for iso3 in GRID3_ISO3_CODES:
    d = grid3_scratch(iso3)
    if d.exists():
        files = sorted(d.glob("*.fgb"))
        size_mb = sum(f.stat().st_size / 1024 / 1024 for f in files)
        fgb_total += len(files)
        fgb_size_mb += size_mb
        print(f"  {iso3}/  ({len(files)} files, {size_mb:.1f} MB)")
        for f in files:
            print(f"    {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

# PMTiles output — recurse into iso3 subdirs
pmtiles_files = sorted(OUTPUT_GRID3_DIR.rglob("*.pmtiles")) if OUTPUT_GRID3_DIR.exists() else []
pmtiles_size_mb = sum(f.stat().st_size / 1024 / 1024 for f in pmtiles_files)
print(f"\nPMTiles (3-pmtiles/grid3/): {len(pmtiles_files)} files")
for f in pmtiles_files:
    rel = f.relative_to(OUTPUT_GRID3_DIR)
    print(f"  {rel} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

print(f"\n{'='*50}")
print(f"DISK USAGE SUMMARY")
print(f"{'='*50}")
print(f"  Overture inputs:   {overture_size_mb:.1f} MB ({len(overture_files)} files)")
print(f"  FlatGeobuf:        {fgb_size_mb:.1f} MB ({fgb_total} files)")
print(f"  PMTiles:           {pmtiles_size_mb:.1f} MB ({len(pmtiles_files)} files)")
print(f"  Total:             {overture_size_mb + fgb_size_mb + pmtiles_size_mb:.1f} MB")

## Convert GeoParquet to FlatGeobuf

Convert downloaded Overture GeoParquet files to FlatGeobuf format for tippecanoe compatibility

**Note**: ArcGIS data was already downloaded as FlatGeobuf in Step 2a, so this step only processes Overture Maps data. Both sources will coexist in the scratch directory.

### Disk Space Management

**Automatic cleanup enabled**: Source `.parquet` files are automatically removed after successful conversion to save disk space.

**Why this works:**
- **FlatGeobuf files (.fgb)** are retained as intermediary files
- PMTiles can be quickly regenerated from .fgb files anytime
- .fgb files are ~30-50% smaller than .parquet
- Conversion from .fgb -> .pmtiles is faster than .parquet -> .pmtiles


**To disable autoremove cleanup:** Set `cleanup_source=False` in the conversion step below.

In [ ]:
print("\n" + "="*70)
print("STEP 3: CONVERTING OVERTURE GEOPARQUET TO FLATGEOBUF")
print("="*70)
print(f"  Input:   {OVERTURE_INPUT_DIR}")
print(f"  Output:  {SCRATCH_OVERTURE_DIR}")
print(f"  Cleanup: {'ENABLED' if CONFIG['fgb_conversion']['cleanup_source'] else 'DISABLED'}")

fgb_results = batch_convert_directory(
    input_dir=str(OVERTURE_INPUT_DIR),
    output_dir=str(SCRATCH_OVERTURE_DIR),
    pattern=CONFIG["fgb_conversion"]["input_pattern"],
    overwrite=CONFIG["fgb_conversion"]["overwrite"],
    verbose=CONFIG["fgb_conversion"]["verbose"],
    cleanup_source=CONFIG["fgb_conversion"]["cleanup_source"]
)

overture_fgb = sorted(SCRATCH_OVERTURE_DIR.glob("*.fgb")) if SCRATCH_OVERTURE_DIR.exists() else []
print("="*70)
print(f"  Overture FGBs in overture/: {len(overture_fgb)}")
if CONFIG["fgb_conversion"]["cleanup_source"]:
    print(f"  Source .parquet files removed to save disk space")
for f in overture_fgb[:10]:
    print(f"    • {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")
if len(overture_fgb) > 10:
    print(f"    ... and {len(overture_fgb) - 10} more")
print("="*70)

## Convert Local Geospatial Data to FlatGeobuf 

For local data sources (feature classes, shapefiles, geopackages, etc.) that don't need API download

In [ ]:
print("=== STEP 3a: CONVERTING LOCAL GEOSPATIAL DATA (OPTIONAL) ===\n")

# ── GeoPackage → NGA scratch ─────────────────────────────────────────────────
convert_gpkg_to_fgb_layers(
    gpkg_path=str(grid3_input("nga") / "GRID3_NGA_roads_2163171622446144938.gpkg"),
    output_dir=str(grid3_scratch("nga")),
)

# ── Shapefile → COD scratch ──────────────────────────────────────────────────
# convert_geodata_to_fgb(
#     input_path="/path/to/shapefile.shp",
#     output_path=str(grid3_scratch("cod") / "my_layer.fgb"),
#     clip_extent=CONFIG["extent"]["coordinates"],
#     overwrite=True,
#     verbose=True
# )

# ── ArcGIS File Geodatabase → NGA scratch ───────────────────────────────────
# convert_geodata_to_fgb(
#     input_path=str(grid3_input("nga") / "GRID3_NGA_settlement_extents_v3_1.gdb"),
#     output_path=str(grid3_scratch("nga") / "GRID3_NGA_settlement_extents_v3_1.fgb"),
#     layer="GRID3_NGA_settlement_extents_v3_1",
#     clip_extent=None,
#     overwrite=True,
#     verbose=True
# )

# ── GeoPackage layers → COD scratch ──────────────────────────────────────────
# convert_geodata_to_fgb(
#     input_path=str(grid3_input("cod") / "GRID3_COD_IT_geospatial_data_layers_20260611.gpkg"),
#     output_path=str(grid3_scratch("cod") / "GRID3_COD_IT_zonesante_20260611.fgb"),
#     layer="zonesante",
#     overwrite=True,
#     verbose=True
# )

# ── Batch convert ─────────────────────────────────────────────────────────────
# batch_results = batch_convert_geodata(
#     input_paths=[grid3_input("cod") / "file1.shp", grid3_input("cod") / "file2.geojson"],
#     output_dir=str(grid3_scratch("cod")),
#     clip_extent=CONFIG["extent"]["coordinates"],
#     overwrite=True,
#     verbose=True
# )

print("✓ Step 3a complete\n")
print(f"  COD scratch: {grid3_scratch('cod')}")
print(f"  NGA scratch: {grid3_scratch('nga')}")

### Step 3a-dissolve: Dissolve IT Health Zones → Antenne & Province

Dissolve Ituri health zone boundaries into two derived admin levels using DuckDB spatial unions.

- **Antenne** — grouped by `antenne`; retains `pays, iso3, province, prov_uid, date, edit_par, grid3id`
- **Province** — grouped by `province`; retains `pays, iso3, prov_uid, date, edit_par, grid3id`

Output files are written to `GRID3_boundaries/` so the merge step (3c) can find them alongside the v8_0 equivalents.

In [ ]:
import subprocess, sys
print("=== STEP 3a-dissolve: DISSOLVING IT HEALTH ZONES → ANTENNE & PROVINCE ===\n")

dissolve_script = str(SCRIPTS_DIR / "dissolveAdmin_nested_duckdb.py")

src_hz       = str(grid3_scratch("cod") / "GRID3_COD_IT_zonesante_20260611.fgb")
dst_antenne  = str(grid3_scratch("cod") / "GRID3_COD_IT_antenne_20260611.fgb")
dst_province = str(grid3_scratch("cod") / "GRID3_COD_IT_province_20260611.fgb")

print(f"  Input : {src_hz}")
print(f"  → Antenne  : {dst_antenne}")
print(f"  → Province : {dst_province}\n")

subprocess.run(
    [sys.executable, dissolve_script, src_hz, dst_antenne, dst_province],
    check=True,
)

print("\n✓ Step 3a-dissolve complete")

## Generate Centroids for Administrative Polygons

Generate interior centroid points for health zones and health areas. These will be used for single-label-per-polygon rendering in the map viewer (interior labels at lower zoom levels).

In [ ]:
# Generate centroids for administrative boundary labels
print("=== STEP 2b: GENERATING CENTROIDS FOR ADMINISTRATIVE LABELS ===")

# Boundary polygon layers that need interior centroids for label placement.
# Centroids are written alongside their source polygons in the same scratch dir.
layers_for_centroids = [
    # COD
    'GRID3_COD_province_v8_0',
    'GRID3_COD_antenne_v8_0',
    'GRID3_COD_zonesante_v8_0',
    'GRID3_COD_airesante_v8_0',
    'GRID3_COD_IT_province_20260611',
    'GRID3_COD_IT_antenne_20260611',
    'GRID3_COD_IT_zonesante_20260611',
    'GRID3_COD_IT_airesante_20260611',
    # NGA
    'GRID3_NGA_national_boundary_unpublished_20260429'
]

centroid_results = batch_generate_centroids(
    input_dir=str(grid3_scratch("cod")),
    output_dir=str(grid3_scratch("cod")),
    layers=layers_for_centroids,
    suffix='_centroids',
    verbose=CONFIG["download"]["verbose"]
)

print(f"\nCentroid Generation Summary:")
print(f"  Total layers: {centroid_results['total_layers']}")
print(f"  Successful:   {centroid_results['successful']}")
print(f"  Failed:       {centroid_results['failed']}")

for layer in centroid_results['layers']:
    if layer['success']:
        output_name = Path(layer['output_file']).name
        print(f"  ✓ {output_name}: {layer['feature_count']:,} centroids")
    else:
        print(f"  ✗ {Path(layer['input_file']).name}: {layer.get('error', 'Unknown error')}")

### Step 3b: Filter v8_0 Files (Remove superseded layer(s))

Strip province-X features from the full DRC v8_0 FGB files and write filtered copies to `_filtered/` subdirectories. This is done before the merge step so the Ituri-only (IT) dataset can be appended cleanly without duplication.

Rules are defined in `utilities/ituri_filter_rules.json` — each entry maps a v8_0 filename to a SQL WHERE clause for features **to delete**.

In [ ]:
print("=== STEP 3b: FILTERING v8_0 FILES (REMOVE ITURI) ===")
import json, duckdb

filter_config = json.loads((SCRIPTS_DIR / "ituri_filter_rules.json").read_text())
con = duckdb.connect(); con.install_extension("spatial"); con.load_extension("spatial")

# Filter COD files: v8_0 boundaries and POIs both live in grid3_scratch("cod")
# Filtered copies go to _filtered/ subdir so the merge step can find them cleanly
for src_dir, out_dir in [
    (grid3_scratch("cod"), grid3_scratch_filtered("cod")),
]:
    out_dir.mkdir(parents=True, exist_ok=True)
    for fgb in sorted(src_dir.glob("*.fgb")):
        where = _resolve_filter(fgb.name, filter_config, None)
        if where:
            _filter_fgb(fgb, out_dir / fgb.name, where, con)

con.close()
print(f"\n✓ Step 3b complete — filtered files in {grid3_scratch_filtered('cod')}")

### Step 3c: Merge Filtered v8_0 + IT Files

Combine each filtered v8_0 file with its Ituri (IT) counterpart into a single FlatGeobuf using DuckDB `UNION ALL BY NAME`. The merged file overwrites the v8_0 path in the thematic scratch directory so tippecanoe reads a single coherent dataset per layer.

A `merge_src` field (`"v8_0"` or `"IT_20260611"`) is added to every row for provenance tracking. Schema differences (v8_0 has extra columns like `OBJECTID`, `source_acronym`) are handled gracefully — missing columns in IT rows become `NULL`.

Rules are defined in `utilities/cod_merge_rules.json`.

In [ ]:
print("=== STEP 3c: MERGING FILTERED v8_0 + IT FILES ===")

# cod_merge_rules.json references files relative to the COD scratch dir
results = run_merge_config(
    config_path=SCRIPTS_DIR / "cod_merge_rules.json",
    scratch_dir=grid3_scratch("cod"),
    verbose=True,
)

ok   = [r for r in results if r["success"]]
fail = [r for r in results if not r["success"]]
print(f"\nMerge: {len(ok)} OK, {len(fail)} failed | "
      f"Total features: {sum(r.get('n_total', 0) for r in ok):,}")

if fail:
    for r in fail:
        print(f"  ✗ {Path(r['output']).name}: {r.get('error', '?')}")

## Process FlatGeobuf to PMTiles

Use the `runCreateTiles.py` module to convert FlatGeobuf files to PMTiles using custom tippecanoe queries from tippecanoe.py

### Process an individual layer theme for a single country

In [ ]:
# (a) Single LAYER_GROUP for one ISO3
# Searches only the matching ISO3 scratch dir; output goes to OUTPUT_GRID3_DIR/{iso3}/.
# Set the group name to any key from LAYER_GROUPS (profiles/{iso3}/layers.yaml).

TARGET_GROUP = "GRID3_NGA_roads"   # < - change as needed

results_a = process_group(
    TARGET_GROUP,
    tiling_profile=CONFIG["tiling"]["profile"],
    verbose=CONFIG["tiling"]["verbose"],
)

if results_a.get("errors"):
    print(f"Errors: {results_a['errors']}")
else:
    print(f"\n✓ Processed: {[e['output_file'] for e in results_a.get('processed_files', [])]}")

### Process an individual layer theme for all available countries

In [ ]:
# (b) One theme across all available countries
# Discovers GRID3_{ISO3}_{theme} groups for every ISO3 whose scratch dir has the files.
# Searches all ISO3 scratch dirs; output goes to OUTPUT_GRID3_DIR (root grid3/ level).
# Available themes: "boundaries", "settlement_extents", "roads", "POIs"

TARGET_THEME = "boundaries"   # < - change as needed

results_b = process_theme(
    TARGET_THEME,
    tiling_profile=CONFIG["tiling"]["profile"],
    verbose=CONFIG["tiling"]["verbose"],
)

if results_b.get("errors"):
    print(f"Errors: {results_b['errors']}")
else:
    files = [e['output_file'] for e in results_b.get('processed_files', [])]
    print(f"\n✓ {len(files)} archive(s) written: {[str(f) for f in files]}")

### Process all layer themes for an individual country

In [ ]:
# (c) All LAYER_GROUPs for one country
# Narrows the file search to that country's scratch dir automatically.
# Available ISO3 codes: GRID3_ISO3_CODES (e.g. "cod", "nga")

TARGET_ISO3 = "nga"   # < - change as needed

results_c = process_iso3(
    TARGET_ISO3,
    tiling_profile=CONFIG["tiling"]["profile"],
    verbose=CONFIG["tiling"]["verbose"],
)

if results_c.get("errors"):
    print(f"Errors: {results_c['errors']}")
else:
    files = [e['output_file'] for e in results_c.get('processed_files', [])]
    print(f"\n✓ {len(files)} archive(s) written for {TARGET_ISO3.upper()}")

=== PROCESSING TO TILES ===
  Note: 1 file(s) not in any LAYER_GROUP — skipped:
    GRID3_NGA_roads_v1_0.fgb
Found 3 file(s) assigned to groups (1 unmatched, skipped):
  GRID3_NGA_roads_v1_0_4326.fgb (FlatGeobuf)
  GRID3_NGA_settlement_extents_dissolve_v4_0.fgb (FlatGeobuf)
  GRID3_NGA_settlement_extents_v4_0.fgb (FlatGeobuf)
  → group 'GRID3_NGA_settlement_extents': GRID3_NGA_settlement_extents.pmtiles
  → group 'GRID3_NGA_POIs': GRID3_NGA_POIs.pmtiles
  → group 'GRID3_NGA_roads': GRID3_NGA_roads.pmtiles

Processing group 'GRID3_NGA_settlement_extents'...


detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
4787628 features, 1847966622 bytes of geometry and attributes, 84702745 bytes of string pool, 17574508320 bytes of vertices, 54109336 bytes of nodes
Choosing a maxzoom of -z9 for features typically 688 feet (210 meters) apart, and at least 166 feet (51 meters) apart
Choosing a maxzoom of -z15 for resolution of about 12 feet (3 meters) within features
tile 7/68/59 size is 2199209 with detail 12, >2097152    
Going to try keeping the sparsest 76.29% of the features to make it fit
tile 7/65/59 size is 3057244 with detail 12, >2097152    
Going to try keeping the sparsest 54.88% of the features to make it fit
tile 7/65/60 has 200001 (estimated 212383) features, >200000    
Going to try keeping the sparsest 75.34% of the features to make it fi

✓ GRID3_NGA_settlement_extents → /tmp/grid3_tiles/data/2-scratch/grid3/nga/_temp/GRID3_NGA_settlement_extents.pmtiles

Processing group 'GRID3_NGA_POIs'...
✓ GRID3_NGA_POIs → /tmp/grid3_tiles/data/2-scratch/grid3/nga/_temp/GRID3_NGA_POIs.pmtiles

Processing group 'GRID3_NGA_roads'...


detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
43924755 features, 4715642349 bytes of geometry and attributes, 158625613 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
tile 5/17/14 has 198601 (estimated 1700179) features, >200000    
Going to try keeping the sparsest 9.41% of the features to make it fit
tile 5/17/15 has 198468 (estimated 1729737) features, >200000    
Going to try keeping the sparsest 9.25% of the features to make it fit
tile 5/17/14 has 199155 (estimated 204207) features, >200000    
Going to try keeping the sparsest 7.37% of the features to make it fit
tile 5/17/15 ha

### Process all layer themes for all available countries

In [ ]:
# Step 4: Process all geospatial files to PMTiles
print("=== STEP 4: PROCESSING TO PMTILES ===")
print(f"Tiling profile:  {CONFIG['tiling']['profile']}")
print(f"Input dirs:")
for d in CONFIG["tiling"]["input_dirs"]:
    print(f"  {d}")
print(f"Output root:     {CONFIG['tiling']['output_dir']}")

tiling_results = process_to_tiles(
    extent=None,
    # extent=CONFIG["extent"]["coordinates"],
    input_dirs=[str(d) for d in CONFIG["tiling"]["input_dirs"]],
    filter_pattern=CONFIG["tiling"]["filter_pattern"],
    output_dir=str(CONFIG["tiling"]["output_dir"]),
    parallel=CONFIG["tiling"]["parallel"],
    verbose=CONFIG["tiling"]["verbose"],
    tiling_profile=CONFIG["tiling"]["profile"],
    keep_theme_files=CONFIG["tiling"].get("keep_theme_files", True),
)

if tiling_results["errors"]:
    print(f"\nErrors encountered: {len(tiling_results['errors'])}")
    for error in tiling_results["errors"]:
        print(f"  - {error}")

# List generated PMTiles — recurse into iso3 subdirs
pmtiles_files = sorted(OUTPUT_GRID3_DIR.rglob("*.pmtiles")) if OUTPUT_GRID3_DIR.exists() else []
if pmtiles_files:
    total_size_mb = 0
    print(f"\nPMTiles in 3-pmtiles/grid3/:")
    for pmtile in pmtiles_files:
        size_mb = pmtile.stat().st_size / 1024 / 1024
        total_size_mb += size_mb
        rel = pmtile.relative_to(OUTPUT_GRID3_DIR)
        print(f"  {rel} ({size_mb:.1f} MB)")
    print(f"\nTotal: {total_size_mb:.1f} MB across {len(pmtiles_files)} file(s)")
else:
    print("\nNo PMTiles found. Check errors above.")
    print(f"  Expected output: {OUTPUT_GRID3_DIR}")

## Step 4b: Standalone tile-join merge (iso3 / all)

Re-merge existing per-theme PMTiles into coarser archives without re-running the full tippecanoe Step 4.
Run this after retiling any subset of groups to refresh `GRID3_{ISO3}.pmtiles` or `GRID3.pmtiles`.

- **`MERGE_SOURCE_DIR`** — override to a specific directory; `None` = auto-detect from
  `2-scratch/grid3/{iso3}/_temp/` (written by Step 4 when profile is `"iso3"` or `"all"`)
  or fall back to `3-pmtiles/grid3/{iso3}/`.
- **`MERGE_PROFILE`** — `"iso3"` or `"all"`. Defaults to `CONFIG["tiling"]["profile"]`.
- **`MERGE_KEEP_SOURCE`** — set `False` to remove `_temp/` dirs after merging.

In [2]:
import subprocess, json
from pathlib import Path
from collections import defaultdict

# ── Configuration ─────────────────────────────────────────────────────────────
# MERGE_SOURCE_DIR: country-specific output dir when available (e.g. grid3/cod)
# Falls back to auto-detect when no country subdir with PMTiles is found.
MERGE_PROFILE     = CONFIG["tiling"]["profile"]   # "iso3" or "all"
MERGE_KEEP_SOURCE = CONFIG["tiling"].get("keep_theme_files", True)

MERGE_SOURCE_DIR = next(
    (
        OUTPUT_GRID3_DIR / iso3
        for iso3 in GRID3_ISO3_CODES
        if (OUTPUT_GRID3_DIR / iso3).exists()
        and any((OUTPUT_GRID3_DIR / iso3).glob("*.pmtiles"))
    ),
    None,
)

# ── Resolve source archives ───────────────────────────────────────────────────
if MERGE_SOURCE_DIR:
    theme_files = sorted(Path(MERGE_SOURCE_DIR).glob("*.pmtiles"))
    src_label   = str(MERGE_SOURCE_DIR)
else:
    # Collect per-theme archives from iso3 scratch _temp dirs (written by Step 4
    # when profile is "iso3" or "all"), falling back to the iso3 output subdirs.
    theme_files = []
    for iso3 in GRID3_ISO3_CODES:
        _temp = grid3_scratch_temp(iso3)
        _out  = grid3_output(iso3)
        if _temp.exists() and any(_temp.glob("*.pmtiles")):
            theme_files.extend(sorted(_temp.glob("*.pmtiles")))
        elif _out.exists():
            theme_files.extend(sorted(_out.glob("*.pmtiles")))
    src_label = "per-ISO3 scratch/_temp or output dirs"

print(f"Source: {src_label}")
print(f"Profile: {MERGE_PROFILE}")

if not theme_files:
    print("No .pmtiles source files found.")
else:
    print(f"  {len(theme_files)} archive(s):")
    for f in theme_files:
        print(f"    {f.relative_to(f.parents[2]) if f.parents[2].exists() else f.name}"
              f" ({f.stat().st_size/1024/1024:.1f} MB)")

    def _iso3(stem):
        parts = stem.split("_")
        return parts[1] if len(parts) >= 3 else stem

    def _maxzoom(path):
        try:
            out = subprocess.check_output(["pmtiles", "show", "--json", str(path)], stderr=subprocess.DEVNULL)
            return json.loads(out).get("maxzoom")
        except Exception:
            return None

    def _join(output_path, inputs):
        zooms = [z for z in (_maxzoom(p) for p in inputs) if z is not None]
        cmd = ["tile-join", "-fo", str(output_path), "--no-tile-size-limit"]
        if zooms:
            cmd += [f"-z{max(zooms)}"]
        cmd += [str(p) for p in inputs]
        subprocess.run(cmd, check=True)
        size_mb = output_path.stat().st_size / 1024 / 1024
        print(f"  ✓ {output_path.name} ({size_mb:.1f} MB)")
        return output_path

    if MERGE_PROFILE == "iso3":
        by_iso3 = defaultdict(list)
        for p in theme_files:
            by_iso3[_iso3(p.stem)].append(p)
        print(f"\nMerging {len(theme_files)} archive(s) into {len(by_iso3)} country file(s)...")
        for iso3, inputs in sorted(by_iso3.items()):
            # iso3-merged files land at the grid3/ level, not inside the iso3 subdir
            _join(OUTPUT_GRID3_DIR / f"GRID3_{iso3}.pmtiles", inputs)

    elif MERGE_PROFILE == "all":
        print(f"\nMerging {len(theme_files)} archive(s) into GRID3.pmtiles...")
        _join(OUTPUT_GRID3_DIR / "GRID3.pmtiles", theme_files)

    else:
        print(f"Profile '{MERGE_PROFILE}' is 'merge profile set to default, no merge needed.")

    if not MERGE_KEEP_SOURCE:
        import shutil
        cleaned = set()
        for f in theme_files:
            d = f.parent
            if d.name == "_temp" and d not in cleaned:
                shutil.rmtree(d, ignore_errors=True)
                cleaned.add(d)
        if cleaned:
            print(f"\nRemoved {len(cleaned)} _temp dir(s)")

Source: /tmp/grid3_tiles/data/3-pmtiles/grid3/cod
Profile: iso3
  3 archive(s):
    grid3/cod/GRID3_COD_POIs.pmtiles (227.7 MB)
    grid3/cod/GRID3_COD_boundaries.pmtiles (276.8 MB)
    grid3/cod/GRID3_COD_settlement_extents.pmtiles (296.9 MB)

Merging 3 archive(s) into 1 country file(s)...


  ✓ GRID3_COD.pmtiles (825.2 MB)


## Step 4c: Assemble Africa-level Archives

Tile-join all per-ISO3 per-theme PMTiles into continent-level archives, then merge
those into a single `GRID3_africa.pmtiles` for use as the webmap source.

Assembly rules live in [profiles/africa/assembly.yaml](profiles/africa/assembly.yaml) —
edit that file to control which themes are included and in what draw order.

- Per-theme outputs → `3-pmtiles/grid3/africa/GRID3_africa_{theme}.pmtiles`
- Final merged archive → `3-pmtiles/grid3/africa/GRID3_africa.pmtiles`

Run with `dry_run=True` first to confirm sources are auto-discovered correctly.

In [ ]:
import sys
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from assemble_africa import assemble_africa

print("=== STEP 4c: AFRICA ASSEMBLY ===")

# dry_run=True previews discovered sources without running tile-join
africa_results = assemble_africa(
    output_dir=OUTPUT_GRID3_DIR,
    dry_run=True,       # set False to execute
    verbose=True,
)

# Summary
ok   = [r for r in africa_results if r["success"] and not r["skipped"]]
skip = [r for r in africa_results if r["skipped"]]
fail = [r for r in africa_results if not r["success"] and not r["skipped"]]
print(f"\nResults: {len(ok)} assembled, {len(skip)} skipped (no sources), {len(fail)} failed")

## Step 5: Migrate PMTiles to Persistent Storage

`/tmp/grid3_tiles` is wiped on WSL2 reboot. Run this step after any tiling session to rsync
the `3-pmtiles` tree to the persistent mirror at `/mnt/d/mheaton/grid3_tiles/data/3-pmtiles`.

- **`dry_run=True`** (default) previews what would be transferred without writing
- **`dry_run=False`** executes the rsync
- `rsync -av --mkpath` preserves directory structure and shows per-file progress
- Only changed/new files are transferred on subsequent runs (rsync delta sync)

In [ ]:
import subprocess
from pathlib import Path

def migrate_pmtiles(
    source: Path = OUTPUT_DIR,
    target: Path = MIGRATION_TARGET_DIR,
    dry_run: bool = True,
) -> bool:
    """Rsync 3-pmtiles from /tmp to persistent /mnt/d storage.

    Args:
        source:  Source directory (default: OUTPUT_DIR = 3-pmtiles root in /tmp).
        target:  Destination directory (default: MIGRATION_TARGET_DIR from config/.env).
        dry_run: If True, show what would be transferred without writing.

    Returns:
        True if rsync exited successfully.
    """
    cmd = ["rsync", "-av", "--info=progress2", "--mkpath", f"{source}/", f"{target}/"]
    if dry_run:
        cmd.insert(1, "--dry-run")

    label = "DRY RUN — " if dry_run else ""
    print(f"=== STEP 5: MIGRATE PMTILES ({label}rsync) ===")
    print(f"  Source: {source}")
    print(f"  Target: {target}")
    print()

    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
    if dry_run:
        print("--- Dry run complete. Set dry_run=False to execute. ---")
    else:
        print("--- Migration complete. ---")
    return result.returncode == 0


# uncomment the second call to execute
# migrate_pmtiles(dry_run=True)
migrate_pmtiles(dry_run=False)

## Step 5b: Generate STAC Item Sidecars

Write a `{archive}.stac.json` sidecar alongside each PMTiles file in the output tree.
Items are STAC 1.0.0 compliant and include bbox, datetime, DOI link, version, and a
`data` asset pointer using the `application/vnd.pmtiles` media type.

Version/DOI metadata is read from the structured JSON block embedded in each archive's
`description` field by tippecanoe (set at tile-generation time via `build_tippecanoe_group_command`).

Sidecars can be ingested directly into any STAC API or validated with `stac-validator`.

In [ ]:
from generate_stac import generate_stac_items

print("=== STEP 5b: STAC ITEM GENERATION ===")

stac_results = generate_stac_items(
    tile_dir=OUTPUT_GRID3_DIR,
    overwrite=False,    # set True to regenerate existing sidecars
    verbose=True,
)

ok   = [r for r in stac_results if r["success"] and not r["skipped"]]
skip = [r for r in stac_results if r["skipped"]]
fail = [r for r in stac_results if not r["success"]]
print(f"\nSidecar files generated: {len(ok)}  |  skipped: {len(skip)}  |  failed: {len(fail)}")

if ok:
    print(f"\nExample sidecar:")
    import json
    sample = json.loads(Path(ok[0]["stac_json"]).read_text())
    print(json.dumps({
        "id": sample["id"],
        "bbox": sample["bbox"],
        "datetime": sample["datetime"],
        "properties": {k: v for k, v in sample["properties"].items() if k in ("title", "version", "doi", "iso3")},
        "assets": {"data": {"type": sample["assets"]["data"]["type"]}},
    }, indent=2))